In [1]:
"""
Process the Frantz et al. (2023) 10-meter products to 250-meter matching the COMBUST

maxwell.cook@colorado.edu
"""

import os, glob, time, sys
import rioxarray as rxr
import subprocess
from osgeo import gdal

# Custom functions
sys.path.append(os.path.join(os.getcwd(),'scripts/'))
from functions import *

# Coordinate system for project (albers)
proj = 'ESRI:102039'

# File paths
basedir = '/Users/max/Library/CloudStorage/OneDrive-Personal/mcook/earth-lab/opp-urban-fuels/'
dataraw = os.path.join(basedir, 'data/spatial/raw/')
datamod = os.path.join(basedir, 'data/spatial/mod/')

print("Ready !")

Ready !


In [2]:
# Open a match grid for reproject-matching
match_grid = os.path.join(dataraw, "surfaces_fuel_v2022/urban_fuel_contemporary.tif")
match_grid = rxr.open_rasterio(match_grid, mask=True, cache=False).squeeze()
print_raster(match_grid)

shape: (11615, 18459)
resolution: (250.0, -250.0)
bounds: (-2356398.7592963483, 269249.2873629322, 2258351.2407036517, 3172999.287362932)
sum: 1493100003328.0
CRS: ESRI:102039



In [3]:
# get the extent
xmin, ymin, xmax, ymax = match_grid.rio.bounds()
xmin

-2356398.7592963483

In [4]:
# Functions
def list_dirs(path):
    """
    :param path:
    :return:
    """
    return glob.glob(os.path.join(path, '*'), recursive=True)


def list_files(path, ext, recursive):
    """
    :param path:
    :param ext:
    :param recursive:
    :return:
    """
    if recursive is True:
        return glob.glob(os.path.join(path, '**', '*{}'.format(ext)), recursive=True)
    else:
        return glob.glob(os.path.join(path, '*{}'.format(ext)), recursive=False)


def resample_grid_gdalwarp(in_file, out_file, extent=None, resample_method='sum', res=250):
    """
    Resample a raster file from 10-meter to 250-meter resolution using gdalwarp.

    :param in_file: The input raster file path.
    :param out_file: The output raster file path.
    :param resampling_method: The resampling method to use ('sum', 'average', etc.).
    :param target_resolution: The target resolution in meters (default is 250).
    :return: None
    """
    try:
        # Construct the gdalwarp command
        command = [
            'gdalwarp',
            '-tr', str(res), str(res),  # Target resolution
            '-r', resample_method,  # Resampling method
            '-of', 'GTiff',  # Output format
            '-co', 'COMPRESS=LZW',  # Compression
            '-te', '-2356398.7592963483', '269249.2873629322', '2258351.2407036517', '3172999.2873629322',
            '-srcnodata', '-9999',  # Set source NoData value
            '-dstnodata', '-9999',  # Ensure destination NoData is properly assigned
            '-ot', 'Float32', # ensure output data type
            '-t_srs', 'EPSG:5070', # output CRS
            in_file,
            out_file
        ]

        # Run the command
        subprocess.run(command, check=True)
        print(f'Successfully resampled {in_file} to {out_file}')

    except subprocess.CalledProcessError as e:
        print(f"Error resampling {in_file}: {e}")
        

print("Functions ready !")

Functions ready !


In [5]:
#############################
# Run for each region/state #
t0 = time.time()

# Loop through files and run the warp
# List of subdirectories for each region
dirs = list_dirs(os.path.join(dataraw,'Frantz_et_al/10m/'))

tif_paths = {}  # dictionary to store the list of tif files for each region
for i in range(len(dirs)):
    d = dirs[i]
    region = os.path.basename(d)  # grab the region name
    print(f'Starting for {region} region ...')
    # list the state directories
    subdirs = list_dirs(d)
    subdirs = [sd for sd in subdirs if os.path.basename(sd).startswith("US_")]
    print(f'There are {len(subdirs)} states in the region ...')

    for sd in subdirs:
        state = os.path.basename(sd)  # get the state name

        vrt_files = list_files(os.path.join(sd,'mosaic/mass/building/total/'), '*.vrt', recursive=True)\
        # vrt_files = list_files(os.path.join(sd,'mosaic/volume/building/'), '*.vrt', recursive=True)
        vrt_files = [v for v in vrt_files if '_building_total.vrt' not in os.path.basename(v)]
        print(f"\tProcessing {len(vrt_files)} .vrt files in {state}")

        # loop through the files and perform the resampling
        for vrt in vrt_files:
            # define the output GeoTIFF
            # out_path = os.path.join(datamod, f'Frantz_et_al/resampled/{region}/{state}/volume/')
            out_path = os.path.join(datamod, f'Frantz_et_al/resampled/{region}/{state}/mass/')
            out_file = os.path.join(out_path, os.path.basename(vrt)[:-4]+"_250m.tif")
            if not os.path.exists(out_path):
                os.makedirs(out_path, exist_ok=True)

            # check if the resampled image exists
            if not os.path.exists(out_file):
                # run the gdalwarp implementation
                resample_grid_gdalwarp(vrt, out_file)
            else:
                print(f"\t\tAlready processed: {os.path.basename(vrt)}")
            
    t1 = (time.time() - t0) / 60
    print(f"\nTotal elapsed time for {sd}: {t1:.2f} minutes.\n")

t2 = (time.time() - t0) / 60
print(f"Total elapsed time: {t2:.2f} minutes.")

Starting for greatplains region ...
There are 5 states in the region ...
	Processing 7 .vrt files in US_SD
		Already processed: mass_building_commercial_industrial_total.vrt
		Already processed: mass_building_singlefamily_total.vrt
		Already processed: mass_building_highrise_total.vrt
		Already processed: mass_building_lightweight_total.vrt
		Already processed: mass_building_skyscraper_total.vrt
		Already processed: mass_building_multifamily_total.vrt
		Already processed: mass_building_commercial_innercity_total.vrt
	Processing 7 .vrt files in US_OK
		Already processed: mass_building_commercial_industrial_total.vrt
		Already processed: mass_building_singlefamily_total.vrt
		Already processed: mass_building_highrise_total.vrt
		Already processed: mass_building_lightweight_total.vrt
		Already processed: mass_building_skyscraper_total.vrt
		Already processed: mass_building_multifamily_total.vrt
		Already processed: mass_building_commercial_innercity_total.vrt
	Processing 7 .vrt files in U

KeyboardInterrupt: 